# 3-class rain intensity CNN training

Loads the balanced 3-class data with `Multiclasse_data_pipeline.py`, trains the CNN and saves the best model to `best_model_3class.keras` and the training history to `training_history_3class.csv`.

Classes:
- `0`: light
- `1`: moderate
- `2`: intense, merging heavy and violent

Setup:
- Grayscale input `(256, 256, 1)`;
- Output layer `Dense(3, softmax)`;
- SpecAugment and focused Mixup on the training set, one-hot labels and batch size 64;
- Only `accuracy` is tracked during training; the per-class analysis is in `Evaluation_3class.ipynb`

In [ ]:
import gc
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Sequential
from Multiclasse_data_pipeline import build_datasets
from tensorflow.keras import backend as K
import inspect
import Multiclasse_data_pipeline

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
CSV_PATH       = "Split70-15-15_3class.csv"
N_CLASSES      = 3
EPOCHS         = 50
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
SEED           = 42

PATIENCE_EARLY_STOPPING = 15
PATIENCE_REDUCE_LR      = 5
LR_REDUCE_FACTOR        = 0.5
LR_MIN                  = 1e-6


In [ ]:
class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        collected = gc.collect()
        print(f"   [GC] {collected} objetos liberados.")


In [ ]:
def categorical_focal_loss(gamma=2.0, alpha=None):
    """
    Categorical focal loss (Lin et al., 2017).
    - gamma: focusing parameter for hard examples.
    - alpha: per-class weights; None = uniform.
    Expects one-hot y_true and softmax probabilities in y_pred.
    """
    if alpha is not None:
        alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    else:
        alpha_tensor = None

    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        ce = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, gamma)
        focal_ce = focal_weight * ce
        if alpha_tensor is not None:
            focal_ce = focal_ce * alpha_tensor
        return tf.reduce_sum(focal_ce, axis=-1)

    return loss

In [ ]:
def ArquiteturaCNN(input_shape=(256, 256, 1), n_classes=N_CLASSES):
    l2_reg = regularizers.l2(0.0005)

    model = Sequential(name="ArquiteturaCNN3class")

    # ---------- L1 ----------
    model.add(layers.Conv2D(32, kernel_size=3, padding="same", kernel_regularizer=l2_reg, input_shape=input_shape))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L2 ----------
    model.add(layers.Conv2D(32, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))

    # ---------- L3 ----------
    model.add(layers.Conv2D(64, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L4 ----------
    model.add(layers.Conv2D(64, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))
    model.add(layers.SpatialDropout2D(0.05))

    # ---------- L5 ----------
    model.add(layers.Conv2D(128, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())

    # ---------- L6 ----------
    model.add(layers.Conv2D(128, kernel_size=3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(pool_size=2))
    model.add(layers.SpatialDropout2D(0.10))

    # ---------- L7 ----------
    model.add(layers.Conv2D(128, 3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))
    model.add(layers.SpatialDropout2D(0.10)) 

    # ---------- L8 ----------
    model.add(layers.Conv2D(128, 3, padding="same", kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))
    model.add(layers.SpatialDropout2D(0.15))

    # ---------- L9: Output ----------
    model.add(layers.GlobalAveragePooling2D())        
    model.add(layers.Dense(64, kernel_regularizer=l2_reg))
    model.add(layers.LeakyReLU(alpha=0.1))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.6))
    model.add(layers.Dense(n_classes, activation="softmax"))

    return model

In [ ]:
# Model construction and compilation

train_ds, val_ds, test_ds = build_datasets(CSV_PATH, batch_size=BATCH_SIZE, label_mode="one_hot", augment_train=True, mixup_train=True)

model = ArquiteturaCNN(input_shape=(256, 256, 1), n_classes=N_CLASSES)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=categorical_focal_loss(gamma=2.0, alpha=[0.4, 0.4, 0.2]),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
    ],
)
model.summary()

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model_3class.keras", monitor="val_loss",
        mode="min", save_best_only=True, verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", mode="min",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min",
        factor=LR_REDUCE_FACTOR, patience=PATIENCE_REDUCE_LR,
        min_lr=LR_MIN, verbose=1
    ),
    tf.keras.callbacks.CSVLogger(filename="training_history_3class.csv"),
    GarbageCollectionCallback(),
]

In [ ]:
src = inspect.getsource(Multiclasse_data_pipeline.build_datasets)
print(src)

In [ ]:
# Mixup sanity check
for images, labels in train_ds.take(5):
    print("Labels shape:", labels.shape)
    print("Primeiras 3 labels:")
    print(labels[:3].numpy())

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
n_epochs_run     = len(history.history["loss"])
best_val_loss    = min(history.history["val_loss"])
best_val_acc_idx = int(np.argmax(history.history["val_accuracy"]))
best_val_acc     = history.history["val_accuracy"][best_val_acc_idx]

print(f"Épocas executadas:   {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:     {best_val_loss:.4f}")
print(f"Melhor val_accuracy: {best_val_acc:.4f} (época {best_val_acc_idx + 1})")
print(f"Modelo salvo em:     best_model_3class.keras")
print(f"Histórico em:        training_history_3class.csv")


In [ ]:
# Learning curves

# Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss (focal_loss γ=2.0 + mixup focado)")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

# Final summary
print("=" * 40)
print("RESUMO FINAL")
print("=" * 40)
print(f"Épocas executadas:   {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:     {best_val_loss:.4f}")
print(f"Melhor val_accuracy: {best_val_acc:.4f} (época {best_val_acc_idx + 1})")
print("=" * 40)
print()
print("PRÓXIMO PASSO: rode o Evaluation.ipynb para a análise per-class")
print("(matriz de confusão 3x3, classification report, ROC one-vs-rest,")
print("e análise por áudio via votação majoritária dos 11 segmentos).")
